# MASHeritage — Topic 3 Lab: V1 Data Engineering Pipeline

**Selected Pathway:** **Pathway A — Document / Text (Unstructured Heritage Corpus)**
**Domain:** Bilingual (English / Bahasa Malaysia) Malaysian World Heritage Assistant
**Author:** MASHeritage Team
**Environment:** Google Colab (Zero-cost, no model API key required)

This notebook executes the end-to-end V1 Data Engineering Pipeline:
1. **Part A:** Trace Topic 2 requirements (FR1, FR2, FR3, FR5, NFR4, RAI1, RAI3) to official data sources, ingest 11 official UNESCO and Malaysian government documents, and inspect representative English and Malay artefacts.
2. **Part B:** Normalize text, compare fixed vs. paragraph-aware chunking strategies, export `data/processed/prepared_text.jsonl`, run validation assertions, and record the Part B lineage manifest `data/week3_manifest.json`.
3. **Part C:** Execute the integrated, modular V1 Data Pipeline (`execute_pipeline`), generate `data/pipeline_v1/documents/prepared_text.jsonl` and `pipeline_manifest.json`, and run automated repeatability smoke tests (Cell 9) confirming cryptographic provenance invariance across runs.


In [4]:
# Cell 1 — REQUIRED: Workspace setup (RUN FIRST)
from pathlib import Path
import os, sys, json, csv, re, hashlib, subprocess, shutil
from datetime import datetime, timezone
import random, numpy as np
random.seed(0)
np.random.seed(0)

BASE_DIR = Path.cwd() / "week3_project"
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases matching course specifications
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Base folder:", BASE_DIR)
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)
print("Pipeline V1 folder:", PIPE_DIR)


Base folder: /content/week3_project
Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed
Pipeline V1 folder: /content/week3_project/data/pipeline_v1


## Step A2 — Clone Repository and Install Dependencies
Connect to the official MASHeritage repository to obtain the source registry (`data/sources.csv`) and ingestion module (`src/ingestion/fetch_sources.py`).


In [5]:
REPO_DIR = Path.cwd() / "masheritage-rag-assistant"
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=False)
else:
    subprocess.run(["git", "clone", "https://github.com/Virocchan/masheritage-rag-assistant.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

!pip -q install requests beautifulsoup4 pypdf
print("Working repository:", REPO_DIR)

Working repository: /content/masheritage-rag-assistant


## Step A3 — Ingest and Verify Source Files
Execute `src/ingestion/fetch_sources.py` to download official public HTML sources (UNESCO World Heritage Centre & Jabatan Warisan Negara) with SHA-256 provenance tracking.


In [6]:
# Stage ingested raw sources into Colab workspace
REPO_RAW = REPO_DIR / "data" / "raw"
COLAB_RAW = RAW_DIR / "sources"
COLAB_RAW.mkdir(parents=True, exist_ok=True)

# Copy pre-extracted official sources from repository
for p in (REPO_RAW / "sources").glob("*.txt"):
    shutil.copy2(p, COLAB_RAW / p.name)
if (REPO_RAW / "source_manifest.json").exists():
    shutil.copy2(REPO_RAW / "source_manifest.json", RAW_DIR / "source_manifest.json")

# Optionally execute ingestion script if extra sources needed
try:
    subprocess.run([sys.executable, "src/ingestion/fetch_sources.py"], timeout=60, check=False)
    for p in (REPO_RAW / "sources").glob("*.txt"):
        shutil.copy2(p, COLAB_RAW / p.name)
except Exception as e:
    print(f"Notice: Live web refresh skipped ({e}); using pre-extracted sources.")

files = sorted(COLAB_RAW.glob("*.txt"))
print(f"\nVerified {len(files)} ingested raw text files:")
for p in files:
    print(f"  • {p.name:<22} bytes: {p.stat().st_size:>6}")
assert len(files) >= 2, "Error: Upload/fetch at least two approved source files."
print("\nIngestion verified successfully.")


Verified 11 ingested raw text files:
  • JWN_ACT_EN.txt         bytes:    436
  • JWN_ACT_MS.txt         bytes:    459
  • JWN_REGISTER_EN.txt    bytes:    519
  • JWN_REGISTER_MS.txt    bytes:    524
  • UNESCO_1012.txt        bytes:  12999
  • UNESCO_1013.txt        bytes:  18806
  • UNESCO_1014.txt        bytes:   9320
  • UNESCO_1223.txt        bytes:  10161
  • UNESCO_1396.txt        bytes:   8805
  • UNESCO_1734.txt        bytes:  11599
  • UNESCO_MY.txt          bytes:   1877

Ingestion verified successfully.


## Step A4 — Inspect Two Representative Source Artefacts
Inspect one representative English source (`UNESCO_1223.txt` - Melaka and George Town) and one representative Bahasa Malaysia source (`JWN_REGISTER_MS.txt` - National Heritage Register) to verify readability, schema, character counts, and language nuances.


In [7]:
inspection_targets = [
    ("UNESCO_1223.txt", "English — UNESCO Melaka & George Town"),
    ("JWN_REGISTER_MS.txt", "Bahasa Malaysia — Jabatan Warisan Negara Register")
]

for filename, desc in inspection_targets:
    p = COLAB_RAW / filename
    if not p.exists():
        print(f"NOT AVAILABLE: {filename} — check source_manifest.json")
        continue
    text = p.read_text(encoding="utf-8")
    print("=" * 60)
    print(f"Target: {filename} ({desc})")
    print(f"Characters: {len(text):,} | Words: {len(text.split()):,}")
    print(f"SHA-256: {hashlib.sha256(p.read_bytes()).hexdigest()}")
    print("-" * 60)
    print(f"Preview (first 400 chars):\n{text[:400]}...")
    print("=" * 60 + "\n")
    assert len(text.strip()) > 200, f"Source {filename} contains insufficient text."



Target: UNESCO_1223.txt (English — UNESCO Melaka & George Town)
Characters: 8,844 | Words: 1,308
SHA-256: cf748edad71cd2cdfb02d591f1fd0fde6ca813e83f0d7a5084cf55e4b529a422
------------------------------------------------------------
Preview (first 400 chars):
Melaka and George Town, Historic Cities of the Straits of Malacca

Description

Maps

Documents

Gallery

Video

Indicators

Melaka and George Town, historic cities of the Straits of Malacca have developed over 500 years of trading and cultural exchanges between East and West in the Straits of Malacca. The influences of Asia and Europe have endowed the towns with a specific multicultural heritage ...

Target: JWN_REGISTER_MS.txt (Bahasa Malaysia — Jabatan Warisan Negara Register)
Characters: 524 | Words: 66
SHA-256: 03a02d5f91c6ecf86c9a92ed1f6dc38d38fb077c2abe29c0414096aa08e7c990
------------------------------------------------------------
Preview (first 400 chars):
Pilihan Bahasa

Tapak Warisan

JABATAN WARISAN NEGARA Aras Mezanin

## Step B1 & B2 — Common Source Selection and Lineage Helpers
Common helper functions `select_source()` and `sha256()` required for unambiguous input selection and cryptographic lineage tracking.


In [8]:
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        p = RAW_DIR / relative_path
        if not p.is_file():
            raise FileNotFoundError(f"Source file not found: {p}")
        return p
    matches = sorted(p for p in RAW_DIR.rglob("*") if p.is_file()
                     and (filename is None or p.name == filename)
                     and (suffixes is None or p.suffix.lower() in suffixes))
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select one explicit source path.")
    return matches[0]

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

lineage_records = []
print("Lineage and selection helpers initialized.")


Lineage and selection helpers initialized.


## Pathway A — Text Cleaning and Chunking Strategy Comparison
Compare two alternative transformation methods on representative document `UNESCO_1223.txt`:
1. **Fixed-character chunking (500 chars)**
2. **Paragraph-aware chunking (max 500 chars, paragraph-aligned)**


In [9]:
def clean_text(s):
    s = s.replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive.")
    return [s[i:i+size].strip() for i in range(0, len(s), size) if s[i:i+size].strip()]

def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive.")
    chunks, current = [], ""
    for part in re.split(r"\n\s*\n", s):
        part = part.strip()
        if not part:
            continue
        if len(part) > size:
            if current:
                chunks.append(current)
                current = ""
            chunks.extend(fixed_chunks(part, size))
        elif len(current) + len(part) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + part
        else:
            if current:
                chunks.append(current)
            current = part
    if current:
        chunks.append(current)
    return chunks

sample_file = select_source("sources/UNESCO_1223.txt")
sample_text = clean_text(sample_file.read_text(encoding="utf-8"))

fixed_res = fixed_chunks(sample_text, 500)
para_res = paragraph_chunks(sample_text, 500)

print(f"Source length: {len(sample_text):,} characters")
print(f"Fixed-chunk count: {len(fixed_res)}")
print(f"Paragraph-aware chunk count: {len(para_res)}")
print("-" * 50)
print("Fixed Chunk 0 Sample:")
print(f"\"{fixed_res[0][:200]}...\"")
print("-" * 50)
print("Paragraph-Aware Chunk 0 Sample:")
print(f"\"{para_res[0][:200]}...\"")
assert fixed_res and para_res


Source length: 8,844 characters
Fixed-chunk count: 18
Paragraph-aware chunk count: 27
--------------------------------------------------
Fixed Chunk 0 Sample:
"Melaka and George Town, Historic Cities of the Straits of Malacca

Description

Maps

Documents

Gallery

Video

Indicators

Melaka and George Town, historic cities of the Straits of Malacca have deve..."
--------------------------------------------------
Paragraph-Aware Chunk 0 Sample:
"Melaka and George Town, Historic Cities of the Straits of Malacca

Description

Maps

Documents

Gallery

Video

Indicators..."


### Preparation Decision
We choose **Paragraph-Aware Chunking (max 500 characters)**:
- **Semantic Boundary Preservation:** Avoids severing sentences, proper nouns, and historical dates in mid-thought.
- **Section Integrity:** Respects official UNESCO criteria and statutory headings.
- **Metadata Association:** Accommodates variable paragraph lengths while preventing oversized tokens from exceeding embedding context windows.


In [10]:
# Execute Part B Preparation across all ingested documents
registry_path = REPO_DIR / "data" / "sources.csv"
registry = {r["id"]: r for r in csv.DictReader(registry_path.open(encoding="utf-8-sig"))}

CHUNK_SIZE = 500
METHOD = "paragraph-aware-char-500"
records = []
lineage_records = []

for src in sorted(COLAB_RAW.glob("*.txt")):
    sid = src.stem
    clean = clean_text(src.read_text(encoding="utf-8"))
    chunks = paragraph_chunks(clean, CHUNK_SIZE)
    if not chunks:
        print("SKIP empty source:", sid)
        continue
    meta = registry.get(sid, {})
    digest = sha256(src)
    for i, chunk in enumerate(chunks, 1):
        records.append({
            "record_id": f"{sid}-{i:04d}",
            "text": chunk,
            "source_id": sid,
            "source": str(src.relative_to(BASE_DIR)),
            "source_url": meta.get("url", ""),
            "publisher": meta.get("publisher", ""),
            "language": meta.get("language", "unknown"),
            "site": meta.get("site", ""),
            "source_sha256": digest,
            "processing": METHOD,
            "chunk_index": i,
            "page": None
        })
    clean_out = PROCESSED_DIR / f"{sid}_clean.txt"
    clean_out.write_text(clean, encoding="utf-8")
    lineage_records.append({
        "source": str(src.relative_to(BASE_DIR)),
        "output": str(clean_out.relative_to(BASE_DIR)),
        "processing": METHOD,
        "source_sha256": digest,
        "output_sha256": sha256(clean_out)
    })
    print(f"{sid:<20} -> {len(clean):>6} chars, {len(chunks):>3} chunks")

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
with prepared_path.open("w", encoding="utf-8") as f:
    for row in records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"\nTotal prepared chunks: {len(records)}")
print(f"Export saved to: {prepared_path}")
assert records and all(r["text"].strip() and r["source_url"] for r in records)


JWN_ACT_EN           ->    436 chars,   1 chunks
JWN_ACT_MS           ->    459 chars,   1 chunks
JWN_REGISTER_EN      ->    519 chars,   2 chunks
JWN_REGISTER_MS      ->    524 chars,   2 chunks
UNESCO_1012          ->  11719 chars,  38 chunks
UNESCO_1013          ->  17218 chars,  50 chunks
UNESCO_1014          ->   8243 chars,  25 chunks
UNESCO_1223          ->   8844 chars,  27 chunks
UNESCO_1396          ->   8769 chars,  30 chunks
UNESCO_1734          ->  10394 chars,  35 chunks
UNESCO_MY            ->   1875 chars,   4 chunks

Total prepared chunks: 215
Export saved to: /content/week3_project/data/processed/prepared_text.jsonl


## Step B3 & B4 — Validate and Generate Part B Lineage Manifest
Validate record completeness, non-emptiness, and unique identifiers, then compile `data/week3_manifest.json`.


In [11]:
exported = [json.loads(line) for line in prepared_path.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(exported) == len(records), "Export count mismatch."
assert all(x["text"].strip() and x["source_sha256"] and x["source_url"] for x in exported), "Record validation failed."
assert len({x["record_id"] for x in exported}) == len(exported), "Duplicate record IDs detected."

manifest_b = {
    "pipeline_version": "week3-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project": "MASHeritage",
    "pathway": "A-document-text",
    "chunk_method": METHOD,
    "chunk_size_chars": CHUNK_SIZE,
    "source_count": len(lineage_records),
    "prepared_chunk_count": len(exported),
    "records": lineage_records,
    "prepared_export": str(prepared_path.relative_to(BASE_DIR)),
    "prepared_export_sha256": sha256(prepared_path)
}

manifest_b_path = BASE_DIR / "data" / "week3_manifest.json"
manifest_b_path.write_text(json.dumps(manifest_b, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Part B Validation PASSED:")
print(f"  • Source documents processed: {len(lineage_records)}")
print(f"  • Prepared chunks: {len(exported)}")
print(f"  • Part B Manifest saved: {manifest_b_path}")


Part B Validation PASSED:
  • Source documents processed: 11
  • Prepared chunks: 215
  • Part B Manifest saved: /content/week3_project/data/week3_manifest.json


---
# PART C — Build and Document the V1 Data Pipeline
In Part C, the preparation logic is formalized into an executable, repeatable V1 Data Pipeline following the required Cell 7, Cell 8A, and Cell 9 architecture.


### Cell 7 — REQUIRED: Common V1 Pipeline Utilities
Defines reusable source selection, pipeline runner (`execute_pipeline`), registration (`register_and_run`), validation, and SHA-256 hashing.


In [12]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B
from pathlib import Path
from datetime import datetime, timezone
import random, numpy as np
random.seed(0)
np.random.seed(0)
import hashlib
import json

BASE_DIR = Path.cwd() / "week3_project"
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)
    if source.is_dir():
        inputs = sorted(p for p in source.iterdir() if p.is_file() and p.suffix.lower() in {".txt", ".pdf"})
    else:
        inputs = [source]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")

    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]

    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")

    validation = {
        "output_count": len(outputs),
        "outputs": [{
            "path": str(p.relative_to(BASE_DIR)),
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p)
        } for p in outputs]
    }

    manifest = {
        "pipeline_version": "v1",
        "pipeline_kind": kind,
        "executed_at_utc": datetime.now(timezone.utc).isoformat(),
        "configuration": configuration,
        "sources": [{
            "path": str(p.relative_to(BASE_DIR)),
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p)
        } for p in inputs],
        "inspection": inspection,
        "processing_method": method,
        "validation": validation,
        "source_output_mapping": mappings
    }

    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return manifest

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    result = runner()
    PIPELINE_RUNS[kind] = {
        "runner": runner,
        "first": result
    }
    return result

print("Cell 7 — Common V1 Pipeline Utilities initialized successfully.")


Cell 7 — Common V1 Pipeline Utilities initialized successfully.


### Cell 8A — CONDITIONAL REQUIRED: Integrated Document Pipeline (Pathway A)
Executes the document ingestion, inspection, paragraph-aware chunking, and validation pipeline for the MASHeritage corpus, saving outputs to `data/pipeline_v1/documents/`.


In [13]:
# Cell 8A — CONDITIONAL REQUIRED: Document Pipeline (Pathway A)
SOURCE_A = resolve_source("sources", folder=True)

CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

def inspect_documents(inputs, cfg):
    total_chars = 0
    samples = []
    for src in inputs:
        text = src.read_text(encoding="utf-8")
        total_chars += len(text)
        samples.append({"name": src.name, "chars": len(text)})
    return {
        "document_count": len(inputs),
        "total_characters": total_chars,
        "samples": samples[:3]
    }

def prepare_documents(inputs, out, cfg):
    target_jsonl = out / "prepared_text.jsonl"
    size = int(cfg["chunk_size"])
    registry_path = REPO_DIR / "data" / "sources.csv"
    registry = {r["id"]: r for r in csv.DictReader(registry_path.open(encoding="utf-8-sig"))}
    all_records = []
    mapping = []

    for src in inputs:
        sid = src.stem
        raw_text = src.read_text(encoding="utf-8")
        cleaned = clean_text(raw_text)
        chunks = paragraph_chunks(cleaned, size)
        meta = registry.get(sid, {})
        digest = sha256_file(src)
        for i, chunk in enumerate(chunks, 1):
            all_records.append({
                "record_id": f"{sid}-{i:04d}",
                "text": chunk,
                "source_id": sid,
                "source": str(src.relative_to(BASE_DIR)),
                "source_url": meta.get("url", ""),
                "publisher": meta.get("publisher", ""),
                "language": meta.get("language", "unknown"),
                "site": meta.get("site", ""),
                "source_sha256": digest,
                "processing": f"paragraph-aware-{size}",
                "chunk_index": i
            })
        mapping.append({
            "source": str(src.relative_to(BASE_DIR)),
            "output": str(target_jsonl.relative_to(BASE_DIR))
        })

    with target_jsonl.open("w", encoding="utf-8") as f:
        for rec in all_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    method = f"clean_text; paragraph_chunks(max_size={size})"
    return [target_jsonl], mapping, method

run_A = register_and_run("documents", SOURCE_A, inspect_documents, prepare_documents, CONFIG_A)
print(f"Cell 8A SUCCESS: Pipeline run completed for {run_A['pipeline_kind']}.")
print(f"Manifest written to: data/pipeline_v1/documents/pipeline_manifest.json")


Cell 8A SUCCESS: Pipeline run completed for documents.
Manifest written to: data/pipeline_v1/documents/pipeline_manifest.json


### Cell 9 — REQUIRED: Repeatability and Provenance Verification
Reruns the registered pipeline independently, comparing input source fingerprints, output fingerprints, and mappings to verify bit-level repeatability.


In [14]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
assert PIPELINE_RUNS, "No pipeline runs registered. Run Cell 8A first."
print("--- Repeatability Verification ---")

for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Source-to-output mapping changed for {kind}"
    print(f"PASS: {kind} — source fingerprints, prepared outputs and mappings match.")

print("\nAll selected pipelines passed repeatability smoke checks.")


--- Repeatability Verification ---
PASS: documents — source fingerprints, prepared outputs and mappings match.

All selected pipelines passed repeatability smoke checks.


## Submission Checklist
- [x] Runnable notebook executes top-to-bottom without manual intervention.
- [x] Ingests and inspects English (`UNESCO_1223`) and Malay (`JWN_REGISTER_MS`) sources.
- [x] Compares fixed vs paragraph-aware chunking with clear empirical evidence.
- [x] Preserves cryptographic lineage in `data/week3_manifest.json`.
- [x] Implements V1 Data Pipeline (`execute_pipeline`) under `data/pipeline_v1/documents/`.
- [x] Passes repeatability verification (Cell 9).
- [x] Documented in `docs/week3_data_inventory.md`, `docs/week3_pipeline_decisions.md`, and `docs/week3_pipeline_diagram.png`.
